In [2]:
import pandas as pd
import numpy as np
import h5py
import os
%matplotlib widget
import matplotlib.pyplot as plt
import import_ipynb
import sys
sys.path.append('C:/Users/wg984/Wolfgang/repos/sleep_research_io')
from sleep_research_functions import sleeplab_to_sleep_research_format, index_to_datetime_sleepdata, load_sleep_data, write_to_hdf5_file


In [8]:
data = pd.read_csv('C:/Users/wg984/Dropbox (Partners HealthCare)/AirGoSleepLabData/ShareWithDavidKuller/AirGo_PSG_TimeAligned/AirGo_Files/AirGo_362.csv')

In [10]:
data = data[['DateTime','accX','accY','accZ','Band','CRCStatus']]


In [12]:
data.to_csv('C:/Users/wg984/Dropbox (Partners HealthCare)/AirGoSleepLabData/ShareWithDavidKuller/AirGo_PSG_TimeAligned/AirGo_Files_FormatBB/AirGo_362.csv', index=False)

In [3]:
airgo_dir = 'C:/Users/wg984/Dropbox (Partners HealthCare)/AirGoSleepLabData/ShareWithDavidKuller/AirGo_PSG_TimeAligned/AirGo_Files/'
psg_dir = 'C:/Users/wg984/Dropbox (Partners HealthCare)/AirGoSleepLabData/ShareWithDavidKuller/AirGo_PSG_TimeAligned/PSG_Files/'
annotations_dir = 'C:/Users/wg984/Dropbox (Partners HealthCare)/AirGoSleepLabData/ShareWithDavidKuller/AirGo_PSG_TimeAligned/Annotations/'
masterlist_path = 'C:/Users/wg984/Dropbox (Partners HealthCare)/AirGoSleepLabData/MasterList_Main_1.23.20.csv'

In [4]:
masterlist = pd.read_csv(masterlist_path)
masterlist = masterlist[masterlist.IncludedInStudy==1]

In [5]:
studyids_to_process = masterlist.SID.apply(lambda x: x.split('Air')[1]).values

In [14]:
# studyids_to_process = [1]
# study_id = str(studyids_to_process[0]).zfill(3)
study_id = studyids_to_process[3:6]

In [6]:
study_id

NameError: name 'study_id' is not defined

In [16]:
# load AirGo
airgo = pd.read_csv(os.path.join(airgo_dir, 'AirGo_'+study_id+'.csv'))
airgo.DateTime = pd.to_datetime(airgo.DateTime)
airgo.set_index('DateTime', inplace=True)
airgo.head()

TypeError: join() argument must be str or bytes, not 'ndarray'

In [10]:
# load 
psg = pd.read_csv(os.path.join(psg_dir, 'PSG_Air'+study_id+'_10Hz.csv'))
psg.DateTime = pd.to_datetime(psg.DateTime)
psg.set_index('DateTime', inplace=True)

In [11]:
psg.head()

,Epoch,CHEST,ABD,SPO2,Stage,Apnea
DateTime,,,,,,
2019-01-24 22:58:18.300,0,7080.485357,-140812.348438,98.000946,5.0,0.0
2019-01-24 22:58:18.400,0,7846.428416,-142703.270367,98.000946,5.0,0.0
2019-01-24 22:58:18.500,0,8839.760822,-136839.018815,98.000946,5.0,0.0
2019-01-24 22:58:18.600,0,9616.118222,-18146.498348,98.000946,5.0,0.0
2019-01-24 22:58:18.700,0,10634.939869,112170.266749,98.000946,5.0,0.0


In [12]:
# load EEG arousal data, not contained in Apnea array:


In [97]:
annotation = pd.read_csv(os.path.join(annotations_dir,'Air'+study_id+'_annotations.csv'))
time1 = pd.to_datetime(annotation.time, format = '%H:%M:%S')
before_midnight = np.where(time1.apply(lambda x: x.hour>12))[0]
assert([0] in before_midnight)
before_midnight = before_midnight.shape[0]
after_midnight = time1.shape[0]-before_midnight
before_midnight = [str(psg.iloc[0].name.date())]*before_midnight
after_midnight = [str(psg.iloc[-1].name.date())]*after_midnight
date_array = before_midnight + after_midnight
annotation.time = pd.to_datetime([x[0] +' ' + x[1] for x in list(zip(date_array, annotation.time))], format='%Y-%m-%d %H:%M:%S')
annotation.drop('epoch', axis=1,inplace=True)

In [98]:
for iPotentialDuplicateRound in range(1,9):
    if sum(annotation.time.duplicated()) == 0: break
    new_time = np.array([x.replace(microsecond=iPotentialDuplicateRound*100000) for x in annotation.time[annotation.time.duplicated()]])
    if new_time.shape[0]==1:
        annotation.time[annotation.time.duplicated()] = new_time[0]
    else:
        annotation.time[annotation.time.duplicated()] = new_time
assert(sum(annotation.time.duplicated()) == 0)

C:\Users\wg984\AppData\Local\Continuum\anaconda3\lib\site-packages\ipykernel_launcher.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  import sys
C:\Users\wg984\AppData\Local\Continuum\anaconda3\lib\site-packages\ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """


In [100]:
annotation[1669:1673].time

1669   2019-01-25 04:26:18.000
1670   2019-01-25 04:26:43.000
1671   2019-01-25 04:26:43.100
1672   2019-01-25 04:26:43.200
Name: time, dtype: datetime64[ns]

In [101]:
annotation.set_index('time', inplace=True)
annotation.columns = ['Annotation']

In [15]:
annotation

,duration,event
time,,
2019-01-23 23:40:37.000,0.002,Montage:01_Standard_Montage_AASM_[01]_Ref
2019-01-23 23:40:38.000,0.002,Recording_Analyzer_-_ECG
2019-01-23 23:40:38.100,0.002,Video_Recording_ON
2019-01-23 23:41:42.000,0.002,CPAP:5_cmH2O
2019-01-23 23:41:52.000,0.002,Impedance_at_10_kOhm
...,...,...
2019-01-24 06:24:15.000,0.002,Nasal_Breath
2019-01-24 06:24:16.000,0.002,Oral_Breath
2019-01-24 06:24:37.000,30.000,Sleep_stage_?


In [ ]:
annotation

In [ ]:
study_id

In [ ]:
data = sleeplab_to_sleep_research_format(airgo, psg, annotation = annotation)

In [18]:
hdr = {
    'study_id': np.int32(study_id),
    'MRN': np.int32(1),
    'fs': np.int32(10),
    'start_datetime': np.array([data.index[0].year,data.index[0].month,data.index[0].day, data.index[0].hour, data.index[0].minute, data.index[0].second, data.index[0].microsecond])
}

In [19]:
hdr

{'study_id': 1,
 'MRN': 1,
 'fs': 10,
 'start_datetime': array([  2019,      1,     23,     21,     50,     29, 300000])}

In [24]:
output_h5_path = 'sleeplabtest.h5'
write_to_hdf5_file(data, output_h5_path , hdr = hdr)

In [26]:
data, hdr = load_sleep_data(output_h5_path, idx_to_datetime = 1, verbose = True)

signals to load: ['ABD', 'Annotation', 'Apnea', 'Band', 'CHEST', 'CRCStatus', 'Epoch', 'MRN', 'SPO2', 'Stage', 'accX', 'accY', 'accZ', 'fs', 'start_datetime', 'study_id']


In [27]:
data

,ABD,Annotation,Apnea,Band,CHEST,CRCStatus,Epoch,SPO2,Stage,accX,accY,accZ
2019-01-23 21:50:29.300,-27507.533203,nan,0,1426.0,-63459.195312,NaN,0,98.0,5,-0.950195,-0.500000,-9.398438
2019-01-23 21:50:29.400,-26657.720703,nan,0,1418.0,-62348.347656,NaN,0,98.0,5,-0.845703,-0.441895,-9.507812
2019-01-23 21:50:29.500,-26868.634766,nan,0,1403.0,-64619.527344,NaN,0,98.0,5,-0.671387,-0.410156,-9.460938
2019-01-23 21:50:29.600,-29196.095703,nan,0,1391.0,-67825.445312,NaN,0,98.0,5,-0.777832,-0.466309,-9.429688
2019-01-23 21:50:29.700,-29730.806641,nan,0,1406.0,-71276.257812,NaN,0,98.0,5,-0.790039,-0.588867,-9.421875
...,...,...,...,...,...,...,...,...,...,...,...,...
2019-01-24 06:05:28.800,-19519.957031,nan,0,686.0,-1305.162964,NaN,989,96.0,5,0.023087,9.718750,-0.659180
2019-01-24 06:05:28.900,-21780.779297,nan,0,683.0,-2528.223877,NaN,989,96.0,5,-0.019745,9.742188,-0.581055
2019-01-24 06:05:29.000,-22823.826172,Sleep_stage_W,0,685.0,-3163.646484,NaN,989,96.0,5,-0.020004,9.710938,-0.649414
2019-01-24 06:05:29.100,-23487.003906,nan,0,682.0,-3130.799561,NaN,989,96.0,5,-0.001347,9.687500,-0.623535
